# 08 · Patterns & Case Studies

> **Module 3 · Orchestration Patterns · Lecture 3.1**

In this notebook you will:

- Build a fan-out / fan-in pipeline and run it in the mini-runner
- Use **dynamic task mapping** (`.expand()` / `.expand_kwargs()`) and model it with `project/core/fanout.py`
- Compare **conditional execution** — `BranchOperator` vs `ShortCircuitOperator` — and pick the right trigger rule for every join
- Wire up the three **cross-DAG mechanisms**: `TriggerDagRunOperator`, `ExternalTaskSensor`, and Datasets
- Implement the **"wait for data"** pattern with `project/core/sensor_conditions.py`
- Work through three case studies: a daily orders ETL, a backfill, and a multi-DAG dependency chain
- Make every pattern **idempotent** with `idempotent_load.py` and `dedup_load.py`

## 1. The pattern catalogue

Orchestration problems repeat. There are perhaps a dozen shapes that cover 95% of what real pipelines do, and knowing which one you are in — by name — is most of the skill.

| # | Pattern | Shape | Airflow mechanism |
|---|---|---|---|
| 1 | **Linear pipeline** | A → B → C | `>>` |
| 2 | **Fan-out / fan-in** | A → {B, C, D} → E | `>>` to multiple, or dynamic mapping |
| 3 | **Conditional execution** | A → (B \| C) | `BranchOperator`, `ShortCircuitOperator` |
| 4 | **Dynamic fan-out** | A → {B₁…Bₙ}, n unknown at parse time | **`.expand()` / `.expand_kwargs()`** (2.3+) |
| 5 | **Cross-DAG, push** | DAG₁ → DAG₂ | `TriggerDagRunOperator` |
| 6 | **Cross-DAG, pull** | DAG₂ → DAG₁ | `ExternalTaskSensor` |
| 7 | **Event-driven** | producer updates dataset → consumer scheduled | **Datasets** (2.4+) |
| 8 | **Wait for data** | A waits for a condition | Sensor (reschedule/deferrable) or Dataset |
| 9 | **Batch + stream hybrid** | stream runs continuously; Airflow runs the batch half | Sensor/Trigger on a stream watermark |
| 10 | **Periodic reconciliation** | every N hours, compare and fix | `schedule="0 */6 * * *"` |

This notebook walks the interesting ones, implements each one in the **mini-runner** that ships with the course, and then ties them together in three worked case studies.

### The one rule that runs through all of it

> **The join is where fan-out goes wrong.**

A task with multiple upstream dependencies uses the default `trigger_rule="all_success"`. If any upstream is *skipped* — which is exactly what a branch does — the join is skipped too, and your pipeline silently produces nothing.

The full trigger-rule table:

| Trigger rule | Runs when | Use for |
|---|---|---|
| `all_success` (default) | every upstream succeeded | the normal case |
| `all_failed` | every upstream failed | a cleanup path |
| `all_done` | every upstream finished, whatever the state | always-run cleanup, or a join after a branch |
| `none_failed` | no upstream failed (skips are fine) | a join after a branch |
| `none_failed_min_one_success` | no upstream failed **and** at least one succeeded | a join after a branch — usually what you want |
| `one_failed` | at least one upstream failed | an alert-on-failure path |
| `none_skipped` | no upstream was skipped | a strict join |

The difference between `none_failed` and `none_failed_min_one_success` matters when **every** branch was skipped. `none_failed` runs the join with no data; `none_failed_min_one_success` skips it. The second is almost always correct.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)

# The pure modules this notebook builds on. None of them import Airflow.
import pipeline_logic
import fanout
import sensor_conditions
import idempotent_load
import dedup_load
import local_runner

for module in (pipeline_logic, fanout, sensor_conditions, idempotent_load,
               dedup_load, local_runner):
    print(f"  {module.__name__:<20} {module.__file__}")

print()
print("The mini-runner knows these DAGs:")
for name in sorted(local_runner.DAGS):
    print(f"  - {name}")

## 2. Fan-out / fan-in, built and run

The classic shape: one upstream task, several parallel branches, one join.

```text
                    ┌─▶ transform_emea ──┐
                    │                    │
  read ──▶ validate ┼─▶ transform_amer ──┼──▶ publish
                    │                    │
                    └─▶ transform_apac ──┘
```

Two ways to build the branches:

1. **Statically** — declare each task by hand. Fine when the count is genuinely fixed and small.
2. **Dynamically** — declare *one* task and let Airflow create one instance per element. This is what `fanout.py` is for.

We will do both, and run them through the mini-runner, which implements dependency ordering, trigger rules, branch skip, retries and XCom — the same subset of Airflow semantics that changes observable behaviour.

First, the fan-out helpers. `regions_to_process` decides *which* branches exist; `load_tasks` produces the payload that `.expand_kwargs()` consumes.

In [ ]:
"""The fan-out / fan-in DAG, expressed in the mini-runner's vocabulary.

Every task here has a one-line counterpart in a real Airflow DAG:

    mini-runner Task            Airflow equivalent
    -----------------           ------------------
    Task(...)                   PythonOperator(...)
    dependencies=[...]          upstream >> downstream
    trigger_rule="none_failed"  trigger_rule=TriggerRule.NONE_FAILED
    branch=True                 BranchOperator
    return value                ti.xcom_push(key="return_value")

The mini-runner has no `.expand()`, so this notebook models dynamic task
mapping explicitly with `run_mapped_task` — a helper that runs the mapped
body once per element and collects the returns, which is precisely the
semantics Airflow gives a mapped task:

  * one task *definition*,
  * N task *instances*, created at run time,
  * the instance's own return value,
  * all returns collected into one list under the unmapped task_id.
"""

from local_runner import DAG, Task

#: The fan-out helpers, from project/core/fanout.py.
from fanout import load_tasks, region_summary_csv, regions_to_process

#: The pure pipeline logic, from project/core/pipeline_logic.py.
from pipeline_logic import summarise_by_region, transform_orders


def read_orders(run, context):
    """Read the raw CSV lines for this run.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The raw CSV lines.
    """
    path = COURSE_ROOT / "sample_data" / "orders.csv"
    return path.read_text(encoding="utf-8").splitlines()


def validate(run, context):
    """Validate the raw rows and push the accepted set downstream.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The accepted rows.
    """
    raw = run.pull("read_orders") or []
    accepted, rejected = transform_orders(raw)
    run.xcom.push("validate", "accepted", accepted)
    run.xcom.push("validate", "rejected", rejected)
    run.xcom.push("validate", "summary", summarise_by_region(accepted))
    return accepted


def list_regions(run, context):
    """Return the regions that produced revenue.

    This is the value that `.expand()` would map over in Airflow.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A sorted list of region codes.
    """
    summary = run.pull("validate", "summary") or {}
    return regions_to_process(summary)


def build_load_tasks(run, context):
    """Return one dict per region, for `.expand_kwargs()`.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A list of dicts with `region` and `partition` keys.
    """
    summary = run.pull("validate", "summary") or {}
    partition = context["ds_nodash"]
    return load_tasks(summary, partition)


def transform_region(run, context, region, partition):
    """Transform one region. This is a mapped task's body.

    In Airflow the mapped parameters arrive in `context["params"]`; here
    they are ordinary arguments, so the body reads the same either way.

    Args:
        run: The run context.
        context: The Jinja context.
        region: The region code.
        partition: The partition decorator being loaded.

    Returns:
        A dict describing what was transformed.
    """
    summary = run.pull("validate", "summary") or {}
    csv_text = region_summary_csv(summary, region)
    return {
        "status": "success",
        "region": region,
        "partition": partition,
        "csv": csv_text,
    }


def run_mapped_task(run, context):
    """Run the mapped body once per element and collect the returns.

    This models Airflow's dynamic task mapping:

      * the elements come from `build_load_tasks` — the list `.expand()`
        or `.expand_kwargs()` would map over;
      * each element produces one task instance with its own return value;
      * the returns are collected into a list — which Airflow stores under
        the *unmapped* task_id, so `xcom_pull(task_ids="load_region")`
        returns the whole list.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The list of per-instance return values.
    """
    task_id = "transform_region"
    elements = run.pull("build_load_tasks") or []
    results = []
    for index, element in enumerate(elements):
        value = transform_region(run, context, **element)
        results.append(value)
        run.xcom.push(task_id, f"instance[{index}]", value)
    # Airflow pushes the collected list under the unmapped task_id.
    run.xcom.push(task_id, "results", results)
    return results


def join_regions(run, context):
    """Join the branches' results.

    In Airflow the join pulls the mapped task's collected list under the
    *unmapped* task_id. The static DAG has no such list, so we fall back
    to the four static branch results.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A summary of the join.
    """
    results = run.pull("transform_region", "results")
    if results is None:
        results = [
            run.pull(task_id)
            for task_id in ("transform_amer", "transform_apac",
                            "transform_emea", "transform_latam")
        ]
    results = [r for r in results if r]
    succeeded = [r for r in results if r.get("status") == "success"]
    return {
        "joined": len(succeeded),
        "regions": [r["region"] for r in succeeded],
    }


def build_static_fanout_dag():
    """Build the static fan-out / fan-in DAG.

    Returns:
        The DAG.
    """
    return DAG(
        "patterns_fanout_static",
        tasks=[
            Task("read_orders", read_orders),
            Task("validate", validate, dependencies=["read_orders"]),
            # --- the fan-out: one task per branch -------------------
            Task("transform_amer", transform_region, dependencies=["validate"],
                 params={"region": "AMER", "partition": "20260115"}),
            Task("transform_apac", transform_region, dependencies=["validate"],
                 params={"region": "APAC", "partition": "20260115"}),
            Task("transform_emea", transform_region, dependencies=["validate"],
                 params={"region": "EMEA", "partition": "20260115"}),
            Task("transform_latam", transform_region, dependencies=["validate"],
                 params={"region": "LATAM", "partition": "20260115"}),
            # --- the fan-in: the join ------------------------------
            Task("join_regions", join_regions,
                 dependencies=["transform_amer", "transform_apac",
                               "transform_emea", "transform_latam"],
                 trigger_rule="none_failed"),
        ],
        schedule="@daily",
        start_date="2026-01-01",
        description="Static fan-out over regions, joined by one task.",
    )


def build_mapped_dag():
    """Build the dynamic-mapping DAG.

    The `transform_region` task is a *single* definition. Its instances
    are created at run time by `run_mapped_task`, from the payload that
    `build_load_tasks` produces — which is the whole point of dynamic task
    mapping.

    Returns:
        The DAG.
    """
    return DAG(
        "patterns_fanout_mapped",
        tasks=[
            Task("read_orders", read_orders),
            Task("validate", validate, dependencies=["read_orders"]),
            Task("list_regions", list_regions, dependencies=["validate"]),
            Task("build_load_tasks", build_load_tasks, dependencies=["validate"]),
            # One definition; N instances, decided at run time.
            Task("transform_region", run_mapped_task,
                 dependencies=["build_load_tasks"]),
            Task("join_regions", join_regions, dependencies=["transform_region"],
                 trigger_rule="none_failed"),
        ],
        schedule="@daily",
        start_date="2026-01-01",
        description="Dynamic fan-out over regions, one task instance each.",
    )


STATIC_FANOUT_DAG = build_static_fanout_dag()
MAPPED_DAG = build_mapped_dag()

print(STATIC_FANOUT_DAG.describe())
print()
print(MAPPED_DAG.describe())

In [ ]:
"""Run both fan-out DAGs and compare their results."""

from local_runner import run_dag

LOGICAL_DATE = "2026-01-15"

print("=" * 70)
print("STATIC FAN-OUT")
print("=" * 70)
static_run = run_dag(STATIC_FANOUT_DAG, LOGICAL_DATE, verbose=True)
print("join result:", static_run.pull("join_regions"))
print()

print("=" * 70)
print("DYNAMIC (MAPPED) FAN-OUT")
print("=" * 70)
mapped_run = run_dag(MAPPED_DAG, LOGICAL_DATE, verbose=True)
print("join result:", mapped_run.pull("join_regions"))
print()

# --- the two must agree -----------------------------------------------
static_join = static_run.pull("join_regions")
mapped_join = mapped_run.pull("join_regions")
assert static_join == mapped_join, f"static {static_join} != mapped {mapped_join}"
assert mapped_join["joined"] == 4, f"expected 4 regions, got {mapped_join}"
print("PASS: static and dynamic fan-out produce identical joins.")
print()

# --- the mapped task's instances ---------------------------------------
elements = mapped_run.pull("build_load_tasks")
print(f"mapped instances created at run time: {len(elements)}")
for index, element in enumerate(elements):
    print(f"  transform_region[{index}]  {element}")
print()
print("In Airflow these appear in the UI as transform_region[0], [1], ...")
print("Each is a separate task instance with its own state and logs.")

## 3. Dynamic task mapping (Airflow 2.3+)

This is the single most important Airflow 2.x feature for DAGs that scale with the data.

### The problem it solves

Before Airflow 2.3, a DAG whose width depends on the data had to be generated at **parse time** — a loop in the DAG file:

```python
# ANTI-PATTERN — never do this
for region in ["AMER", "APAC", "EMEA", "LATAM"]:
    task = PythonOperator(
        task_id=f"process_{region}",
        python_callable=process_region,
        op_kwargs={"region": region},
    )
    read >> task
```

It "works", and it is wrong for four reasons:

1. **The width is fixed at parse time.** The regions are hard-coded. If a fifth region appears, nothing happens — silently.
2. **Every task exists in every run**, even the ones with no data. The DAG looks busy forever.
3. **A DAG file with a loop can generate hundreds of tasks**, which slows the scheduler's parsing of *every* DAG in the environment.
4. **It cannot be conditional.** Every loop iteration is created whether or not it is needed.

### The two forms

```python
from airflow.operators.python import PythonOperator

regions = PythonOperator(
    task_id="list_regions",
    python_callable=list_regions,
)

# `.expand()` — one argument per mapped value, from an XComArg.
process_region = PythonOperator.partial(
    task_id="process_region",
    python_callable=process_one_region,
).expand(region=regions.output)

# `.expand_kwargs()` — multiple arguments at once, from a list of dicts.
process_partition = BigQueryInsertJobOperator.partial(
    task_id="process_partition",
    configuration=LOAD_CONFIG,
).expand_kwargs(build_load_tasks)
```

`fanout.load_tasks` produces exactly the payload `.expand_kwargs()` wants:

```python
[{"region": "AMER", "partition": "20260115"},
 {"region": "APAC", "partition": "20260115"},
 {"region": "EMEA", "partition": "20260115"},
 {"region": "LATAM", "partition": "20260115"}]
```

### The five rules

1. **You must use `.partial()`.** `PythonOperator.partial(task_id=..., python_callable=...)` creates a *template*; `.expand(...)` turns it into a mapped operator. Calling `PythonOperator(...)` directly and then `.expand()` is a `TypeError`.
2. **The mapped arguments must be `XComArg`s** — the output of another task — or literals. You cannot map over an arbitrary Python list; the list must come from a task's return value.
3. **Airflow caps the number of mapped instances.** `max_map_length` defaults to 1024, controlled by `[core] max_map_length` in `airflow.cfg`. Mapping 10,000 items fails with `Cannot map over a list with more than 1024 elements`.
4. **Each mapped instance is a separate task instance** with the same `task_id` and a distinct `map_index`. They show up in the UI as `process_region[0]`, `process_region[1]`, and so on.
5. **The mapped operator's other arguments are shared.** Everything in `.partial()` applies to every instance.

### The output of a mapped task

When a mapped task returns a value, Airflow collects the returns and pushes them as a **list** under the *unmapped* task_id:

```python
results = ti.xcom_pull(task_ids="process_region")        # the whole list
one     = ti.xcom_pull(task_ids="process_region", map_indexes=0)   # one instance
```

### Mapping is not free

Each mapped instance is a separate task instance in the metadata database, with its own state, its own logs, and its own slot. Mapping 10,000 items creates 10,000 task instances. The right granularity is usually "per region", "per customer", "per partition day" — **not "per row"**.

### Mapping and the join

A downstream task of a mapped task waits for *all* the mapped instances, and receives the list of returns. If one mapped instance fails, the join is `upstream_failed` — unless the join carries `trigger_rule=NONE_FAILED_MIN_ONE_SUCCESS`, in which case it runs with the partial results.

### The anti-pattern, measured

Let us put numbers on the loop-versus-mapping comparison.

In [ ]:
"""Measure the loop anti-pattern against dynamic task mapping.

The loop version's width is fixed at parse time and grows with the code.
The mapped version's width is decided by the data at run time.
"""

from fanout import load_tasks, regions_to_process
from pipeline_logic import summarise_by_region, transform_orders

lines = (COURSE_ROOT / "sample_data" / "orders.csv").read_text(
    encoding="utf-8").splitlines()
accepted, _ = transform_orders(lines)
summary = summarise_by_region(accepted)

# --- the loop version: width fixed at parse time ------------------------
LOOP_REGIONS = ["AMER", "APAC", "EMEA", "LATAM"]
loop_width = len(LOOP_REGIONS)

# --- the mapped version: width decided by the data ----------------------
mapped_width = len(load_tasks(summary, "20260115"))

print(f"regions in the data         : {regions_to_process(summary)}")
print(f"loop version width          : {loop_width} tasks (fixed at parse time)")
print(f"mapped version width        : {mapped_width} tasks (decided at run time)")
print()

# --- what happens when a fifth region appears --------------------------
HYPOTHETICAL = ["AMER", "APAC", "EMEA", "LATAM", "ANTARCTICA"]
loop_after = len(HYPOTHETICAL)
mapped_after = len(HYPOTHETICAL)
print("Scenario: a fifth region appears in the source data.")
print(f"  loop version   : {loop_after} tasks — but only if someone edits the DAG file")
print(f"  mapped version : {mapped_after} tasks — automatically, from the data")
print()

# --- the metadata-database cost ---------------------------------------
# Each mapped instance is a task instance with its own row, state and logs.
print("Task instances in the metadata database per run:")
print(f"  loop version   : {loop_width} (every run, whether or not it is needed)")
print(f"  mapped version : {mapped_width} (only the ones the data needs)")
print()

# --- the max_map_length ceiling ---------------------------------------
MAX_MAP_LENGTH = 1024
print(f"Airflow's max_map_length is {MAX_MAP_LENGTH}.")
print(f"  mapping {mapped_width} regions is fine.")
print(f"  mapping per *row* over {len(accepted)} rows would be "
      f"{'OVER' if len(accepted) > MAX_MAP_LENGTH else 'under'} the limit "
      f"and is wrong anyway — map per region, not per row.")
print()

# --- the payload .expand_kwargs() consumes -----------------------------
payload = load_tasks(summary, "20260115")
print("The .expand_kwargs() payload:")
for item in payload:
    print(f"  {item}")

# A payload for .expand_kwargs() must be a list of dicts, and every dict
# must have the same keys — otherwise the mapping fails at run time.
assert isinstance(payload, list)
assert all(isinstance(item, dict) for item in payload)
assert all(set(item) == {"region", "partition"} for item in payload)
assert mapped_width == loop_width, "the sample data has the same four regions"

print()
print("PASS: the payload is a list of uniform dicts, ready for .expand_kwargs().")

## 4. Conditional execution

Two mechanisms, and the difference between them is a common source of confusion.

**`BranchOperator`** — chooses *which downstream task* runs. All other downstream tasks are marked skipped.

```python
branch = BranchOperator(
    task_id="choose_strategy",
    python_callable=choose_strategy,   # returns a list of task_ids
)
```

**`ShortCircuitOperator`** — skips *itself* and everything downstream.

```python
skip = ShortCircuitOperator(
    task_id="skip_if_empty",
    python_callable=lambda **ctx: len(ctx["ti"].xcom_pull(task_ids="extract") or []) > 0,
)
```

| | `BranchOperator` | `ShortCircuitOperator` |
|---|---|---|
| Returns | a list of task_ids to run | a boolean |
| The task itself | succeeds | is **skipped** when the condition is false |
| Downstream | the named branches run, the rest are skipped | everything downstream is skipped |
| Use for | "choose a path" | "stop the whole pipeline" |

The course's `04_branching_and_xcom.py` uses `BranchOperator`, and `choose_strategy` returns a one-element list:

```python
def choose_strategy(**context) -> list[str]:
    ti = context["ti"]
    row_count = int(ti.xcom_pull(task_ids="count_rows") or 0)
    threshold = int(ti.xcom_pull(task_ids="count_rows", key="threshold") or FULL_LOAD_THRESHOLD)
    force = bool(context["params"].get("force_full_load", False))
    chosen = "full_load" if (row_count > threshold or force) else "incremental_load"
    return [chosen]
```

### The three mistakes people make with a join after a branch

1. **Forgetting the trigger rule.** With the default `all_success`, one failed branch means the join never runs and the pipeline's final output is missing entirely.
2. **Pulling from the wrong task_id.** `xcom_pull(task_ids="load_partition")` returns the list; `xcom_pull(task_ids="load_partition[0]")` returns one instance's value.
3. **Assuming a fixed width.** The number of partitions changes with the data. A join that assumes four regions will break when there are five.

We will build both the branching DAG and a short-circuit DAG, and run them with a threshold that forces each branch.

In [ ]:
"""Conditional execution, run in the mini-runner.

Two DAGs are built:

  * `patterns_branch` — a BranchOperator that chooses between two paths,
    with a join that survives a skipped branch.
  * `patterns_shortcircuit` — a ShortCircuitOperator that stops the whole
    pipeline when there is nothing to do.
"""

from local_runner import DAG, Task, TaskSkip

#: The branch decision, from project/core/pipeline_logic.py.
from pipeline_logic import should_run_full_load


def count_rows(run, context):
    """Count the rows available for this run.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The row count.
    """
    raw = run.pull("read_orders") or []
    accepted, _ = transform_orders(raw)
    return len(accepted)


def choose_strategy(run, context):
    """Choose between the full-load and incremental paths.

    The return value is the list of task_ids that should run. Everything
    else downstream is skipped — that is the branch's whole contract.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A one-element list naming the task to run.
    """
    row_count = int(run.pull("count_rows") or 0)
    threshold = int(context["__variables__"].get("full_load_threshold", 1_000_000))
    force = bool(context["params"].get("force_full_load", False))
    chosen = "full_load" if (should_run_full_load(row_count, threshold) or force) \
        else "incremental_load"
    print(f"      rows={row_count} threshold={threshold} force={force} -> {chosen}")
    return [chosen]


def full_load(run, context):
    """The full-load branch.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A description of what was done.
    """
    return "full_load"


def incremental_load(run, context):
    """The incremental branch.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A description of what was done.
    """
    return f"incremental_load:{context['ds_nodash']}"


def join_branches(run, context):
    """The join after the branch.

    `trigger_rule="none_failed"` is what makes this work: a *skipped*
    dependency is not a failure, so the join still runs.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The chosen branch's result.
    """
    return run.pull("full_load") or run.pull("incremental_load") or "nothing"


def short_circuit(run, context):
    """Stop the pipeline when there is nothing to process.

    A `ShortCircuitOperator` skips *itself* and everything downstream.
    Returning False raises `TaskSkip` — the mini-runner's equivalent of
    Airflow's `AirflowSkipException`.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        True when the pipeline should continue.

    Raises:
        TaskSkip: When the pipeline should stop.
    """
    row_count = int(run.pull("count_rows") or 0)
    if row_count == 0:
        print("      no rows — short-circuiting the rest of the DAG")
        raise TaskSkip("no rows to process")
    return True


def after_short_circuit(run, context):
    """A task that only runs when the short circuit did not fire.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A description.
    """
    return "published"


def build_branch_dag():
    """Build the branching DAG.

    Returns:
        The DAG.
    """
    return DAG(
        "patterns_branch",
        tasks=[
            Task("read_orders", read_orders),
            Task("count_rows", count_rows, dependencies=["read_orders"]),
            Task("choose_strategy", choose_strategy, dependencies=["count_rows"],
                 branch=True),
            Task("full_load", full_load, dependencies=["choose_strategy"]),
            Task("incremental_load", incremental_load, dependencies=["choose_strategy"]),
            # The join survives a skipped branch.
            Task("join_branches", join_branches,
                 dependencies=["full_load", "incremental_load"],
                 trigger_rule="none_failed"),
        ],
        schedule="@daily",
        start_date="2026-01-01",
        description="A branch with a resilient join.",
    )


def build_short_circuit_dag():
    """Build the short-circuit DAG.

    Returns:
        The DAG.
    """
    return DAG(
        "patterns_shortcircuit",
        tasks=[
            Task("read_orders", read_orders),
            Task("count_rows", count_rows, dependencies=["read_orders"]),
            Task("check_not_empty", short_circuit, dependencies=["count_rows"]),
            Task("publish", after_short_circuit, dependencies=["check_not_empty"]),
        ],
        schedule="@daily",
        start_date="2026-01-01",
        description="A short circuit that stops the pipeline when idle.",
    )


BRANCH_DAG = build_branch_dag()
SHORT_CIRCUIT_DAG = build_short_circuit_dag()
print(BRANCH_DAG.describe())
print()
print(SHORT_CIRCUIT_DAG.describe())

In [ ]:
"""Run the branch DAG three ways and the short-circuit DAG twice."""

from local_runner import run_dag

LOGICAL_DATE = "2026-01-15"

# ---------------------------------------------------------------------
# 1. The incremental path (threshold above the row count)
# ---------------------------------------------------------------------
print("=" * 70)
print("BRANCH: incremental path (threshold = 1,000,000)")
print("=" * 70)
inc_run = run_dag(BRANCH_DAG, LOGICAL_DATE,
                  variables={"full_load_threshold": 1_000_000}, verbose=True)
print("join result:", inc_run.pull("join_branches"))
assert inc_run.state["full_load"] == "skipped", inc_run.state
assert inc_run.state["incremental_load"] == "success", inc_run.state
assert inc_run.state["join_branches"] == "success", inc_run.state
print("PASS: the un-chosen branch was skipped and the join still ran.")
print()

# ---------------------------------------------------------------------
# 2. The full-load path (threshold below the row count)
# ---------------------------------------------------------------------
print("=" * 70)
print("BRANCH: full-load path (threshold = 100)")
print("=" * 70)
full_run = run_dag(BRANCH_DAG, LOGICAL_DATE,
                   variables={"full_load_threshold": 100}, verbose=True)
print("join result:", full_run.pull("join_branches"))
assert full_run.state["full_load"] == "success", full_run.state
assert full_run.state["incremental_load"] == "skipped", full_run.state
assert full_run.state["join_branches"] == "success", full_run.state
print("PASS: the other branch was skipped and the join still ran.")
print()

# ---------------------------------------------------------------------
# 3. The join without a resilient trigger rule — the classic bug
# ---------------------------------------------------------------------
print("=" * 70)
print("BRANCH: the same DAG with trigger_rule='all_success' (the bug)")
print("=" * 70)
from local_runner import DAG, Task  # noqa: E402  (needed for the broken copy)

buggy = DAG(
    "patterns_branch_buggy",
    tasks=[
        Task("read_orders", read_orders),
        Task("count_rows", count_rows, dependencies=["read_orders"]),
        Task("choose_strategy", choose_strategy, dependencies=["count_rows"],
             branch=True),
        Task("full_load", full_load, dependencies=["choose_strategy"]),
        Task("incremental_load", incremental_load, dependencies=["choose_strategy"]),
        # The bug: the default trigger rule.
        Task("join_branches", join_branches,
             dependencies=["full_load", "incremental_load"]),
    ],
)
buggy_run = run_dag(buggy, LOGICAL_DATE,
                    variables={"full_load_threshold": 100}, verbose=True)
print("join result:", buggy_run.pull("join_branches"))
assert buggy_run.state["join_branches"] == "skipped", buggy_run.state
print("PASS: the join never ran. That is the 'my downstream task silently")
print("      never runs' bug — the DAG looks correct and does nothing.")
print()

# ---------------------------------------------------------------------
# 4. Short circuit: the pipeline stops when there is nothing to do
# ---------------------------------------------------------------------
print("=" * 70)
print("SHORT CIRCUIT: with data")
print("=" * 70)
sc_run = run_dag(SHORT_CIRCUIT_DAG, LOGICAL_DATE, verbose=True)
assert sc_run.state["check_not_empty"] == "success", sc_run.state
assert sc_run.state["publish"] == "success", sc_run.state
print("PASS: the pipeline continued.")
print()

print("=" * 70)
print("SHORT CIRCUIT: without data")
print("=" * 70)


def read_nothing(run, context):
    """Read zero rows, so the short circuit fires.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        An empty list.
    """
    return []


empty_dag = DAG(
    "patterns_shortcircuit_empty",
    tasks=[
        Task("read_orders", read_nothing),
        Task("count_rows", count_rows, dependencies=["read_orders"]),
        Task("check_not_empty", short_circuit, dependencies=["count_rows"]),
        Task("publish", after_short_circuit, dependencies=["check_not_empty"]),
    ],
)
empty_run = run_dag(empty_dag, LOGICAL_DATE, verbose=True)
assert empty_run.state["check_not_empty"] == "skipped", empty_run.state
assert empty_run.state["publish"] == "skipped", empty_run.state
print()
print("PASS: the short circuit skipped itself AND everything downstream.")
print("      Compare with the branch: a branch skips its *siblings*, not itself.")

## 5. Cross-DAG dependencies — three mechanisms, not interchangeable

### 1. `TriggerDagRunOperator` — push

```python
from airflow.operators.trigger_dagrun import TriggerDagRunOperator

trigger_next = TriggerDagRunOperator(
    task_id="trigger_downstream_dag",
    trigger_dag_id="branching_and_xcom_child",
    wait_for_completion=False,
    conf={"parent_run_id": "{{ run_id }}"},
)
```

| Parameter | Meaning |
|---|---|
| `trigger_dag_id` | The DAG to start |
| `wait_for_completion` | If `True`, the task blocks until the other DAG finishes (holding a slot) |
| `conf` | The config dict passed to the other run's `{{ params }}` and `dag_run.conf` |
| `logical_date` | Which logical date to trigger it for |
| `reset_dag_run` | If `True`, clear an existing run and start over |
| `trigger_run_id` | A specific run id to create |
| `wait_for_completion_max_timedelta` | How long to wait when `wait_for_completion=True` |

**The `conf` is how you pass data.** It arrives in the triggered DAG as `{{ dag_run.conf }}` and, if the triggered DAG declares matching `params`, as `{{ params.x }}`.

> **Caveat:** `conf` must be **JSON-serialisable**. It is stored in the metadata database. A `datetime` object raises `TypeError: Object of type datetime is not JSON serializable`.

**Push or wait?** `wait_for_completion=False` is the default and the right choice for a chain: the parent finishes and frees its slot while the child runs. `wait_for_completion=True` turns the parent task into a blocking call that holds a worker slot for the child's entire duration — which is exactly the "long-lived task holds a slot" failure mode.

### 2. `ExternalTaskSensor` — pull

```python
from airflow.sensors.external_task import ExternalTaskSensor

wait_for_parent = ExternalTaskSensor(
    task_id="wait_for_parent_dag",
    external_dag_id="orders_ingest",
    execution_delta=timedelta(hours=2),
    mode="reschedule",
    timeout=3600,
)
```

Use it **only when you cannot restructure the two DAGs**. It couples the consumer to the producer's task ids and its logical date, which makes both DAGs harder to change. The `execution_delta` says how far apart the two runs' logical dates are.

### 3. Datasets — declarative (Airflow 2.4+)

Datasets were called "data-aware scheduling" before 2.4. They are Airflow's declarative answer to "run this DAG when that data changes".

```python
from airflow.datasets import Dataset

ORDERS_DATASET = Dataset("gs://YOUR_BUCKET/datasets/orders")

# The producer: this task *updates* the dataset.
publish = PythonOperator(
    task_id="publish_orders",
    python_callable=publish_orders,
    outlets=[ORDERS_DATASET],       # ← the dataset is declared here
)

# The consumer: this DAG is *scheduled by* the dataset.
with DAG(
    dag_id="consume_orders",
    schedule=[ORDERS_DATASET],       # ← and here
    ...
) as dag:
    ...
```

**How it works.** A dataset has a URI. Any DAG that *updates* that dataset becomes a **producer**. Any DAG that *schedules* on that dataset becomes a **consumer**. When `publish_orders` succeeds, the dataset is recorded as updated, and every consumer is queued for a run.

**Dataset URIs are not templated per run.** The URI is a static identifier, so you use a stable URI and the *update* carries the run's context:

```python
from airflow.datasets.metadata import Metadata   # Airflow 2.9+

publish = PythonOperator(
    task_id="publish_orders",
    python_callable=publish_orders,
    outlets=[
        Metadata(ORDERS_DATASET, extra={"logical_date": "{{ ds }}", "row_count": 1245}),
    ],
)
```

**The critical detail:**

> **A dataset-triggered run is not tied to the producer's logical date.** The consumer runs *now*, with its own logical date. If you need the producer's date, you must pass it — through `params`, through an XCom, or through the dataset's event metadata.

### Choosing between them

| | TriggerDagRunOperator | ExternalTaskSensor | Dataset |
|---|---|---|---|
| Direction | push | pull | declarative |
| Polling | none | yes, every `poke_interval` | none |
| Worker slot | held only if `wait_for_completion=True` | held (or transiently held) | none |
| Timeout | not needed | needed | **not possible** |
| Coupling to task names | no | **yes** | no |
| "The data did not arrive" signal | the child never runs | the sensor times out | **nothing happens** |

> **Prefer `TriggerDagRunOperator` when you control both DAGs**, because it *creates* the run rather than polling for one. Prefer a **Dataset** when the coupling is on *data* rather than on *control flow*. Reach for an **ExternalTaskSensor** only when restructuring is genuinely off the table.

The gap in the Dataset column is real and worth stating plainly: **a Dataset cannot time out.** If the producer stops running, the consumer silently stops running, and nothing reports it. That needs a separate monitoring check — not a sensor.

In [ ]:
"""Model the three cross-DAG mechanisms, executably.

  * `TriggerDagRunOperator` — push. The parent *creates* the child's run.
  * `ExternalTaskSensor`    — pull. The child *polls* for the parent's run.
  * `Dataset`              — declarative. The producer *records an update*,
                              and every consumer of that URI is queued.

The mini-runner has none of these, so we model the *semantics*: what gets
created, what gets observed, and what happens when the producer never
runs.
"""

import json
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Any


@dataclass
class DagRunRecord:
    """A record of one DAG run, in the metadata database.

    Attributes:
        dag_id: The DAG's identifier.
        run_id: The run's identifier.
        logical_date: The run's logical date.
        state: The run's state.
        conf: The config passed to the run.
    """

    dag_id: str
    run_id: str
    logical_date: str
    state: str = "queued"
    conf: dict[str, Any] = field(default_factory=dict)

    def as_json(self):
        """Render the record the way `dag_run.conf` would be stored.

        Returns:
            A JSON document.
        """
        return json.dumps(self.conf, sort_keys=True)


class MetadataDb:
    """A tiny stand-in for Airflow's metadata database.

    It stores DAG runs and dataset update events, which is all the three
    cross-DAG mechanisms need.

    Attributes:
        runs: The DAG runs, keyed by ``(dag_id, logical_date)``.
        dataset_events: The dataset update events, keyed by dataset URI.
    """

    def __init__(self) -> None:
        self.runs: dict[tuple[str, str], DagRunRecord] = {}
        self.dataset_events: dict[str, list[dict[str, Any]]] = {}

    def create_run(self, dag_id, logical_date, conf=None):
        """Create a DAG run — what `TriggerDagRunOperator` does.

        Args:
            dag_id: The DAG to run.
            logical_date: The run's logical date.
            conf: The config for the run.

        Returns:
            The run record.

        Raises:
            ValueError: When a run already exists for that logical date.
        """
        key = (dag_id, logical_date)
        if key in self.runs:
            raise ValueError(f"run already exists for {dag_id} @ {logical_date}")
        run_id = f"manual__{logical_date}T00:00:00+00:00"
        record = DagRunRecord(dag_id, run_id, logical_date, conf=dict(conf or {}))
        self.runs[key] = record
        return record

    def reset_run(self, dag_id, logical_date, conf=None):
        """Create a run, clearing any existing one — `reset_dag_run=True`.

        Args:
            dag_id: The DAG to run.
            logical_date: The run's logical date.
            conf: The config for the run.

        Returns:
            The run record.
        """
        key = (dag_id, logical_date)
        if key in self.runs:
            del self.runs[key]
        return self.create_run(dag_id, logical_date, conf)

    def find_run(self, dag_id, logical_date):
        """Find a run — what `ExternalTaskSensor` polls for.

        Args:
            dag_id: The DAG to look for.
            logical_date: The logical date to look for.

        Returns:
            The run record, or None when there is no such run.
        """
        return self.runs.get((dag_id, logical_date))

    def record_dataset_update(self, uri, *, producer_dag_id, logical_date, extra=None):
        """Record a dataset update — what a task with `outlets` does.

        Args:
            uri: The dataset URI.
            producer_dag_id: The DAG that produced the update.
            logical_date: The producer's logical date.
            extra: Optional event metadata (Airflow 2.9+ `Metadata.extra`).

        Returns:
            The recorded event.
        """
        event = {
            "uri": uri,
            "producer_dag_id": producer_dag_id,
            "producer_logical_date": logical_date,
            "extra": dict(extra or {}),
        }
        self.dataset_events.setdefault(uri, []).append(event)
        return event

    def consumers_for(self, uri):
        """Return the update events for a dataset URI.

        Args:
            uri: The dataset URI.

        Returns:
            The list of events, oldest first.
        """
        return list(self.dataset_events.get(uri, []))


#: The dataset URI shared by the producer and the consumer.
ORDERS_DATASET_URI = "gs://YOUR_BUCKET/datasets/orders"


def trigger_dag_run(db, *, trigger_dag_id, logical_date, conf=None,
                    reset_dag_run=False, wait_for_completion=False):
    """Model `TriggerDagRunOperator`.

    Args:
        db: The metadata database.
        trigger_dag_id: The DAG to trigger.
        logical_date: The logical date to trigger for.
        conf: The config to pass. Must be JSON-serialisable.
        reset_dag_run: Whether to clear an existing run.
        wait_for_completion: Whether the parent blocks.

    Returns:
        A dict describing what happened.

    Raises:
        TypeError: When the conf is not JSON-serialisable.
    """
    # Airflow serialises the conf when it stores it — so a datetime in the
    # conf is a TypeError at trigger time, not at task time.
    json.dumps(conf or {})

    if reset_dag_run:
        record = db.reset_run(trigger_dag_id, logical_date, conf)
    else:
        record = db.create_run(trigger_dag_id, logical_date, conf)

    return {
        "triggered": record.dag_id,
        "run_id": record.run_id,
        "conf": record.as_json(),
        "parent_slot_held": bool(wait_for_completion),
    }


def external_task_sensor_poll(db, *, external_dag_id, logical_date,
                              execution_delta_hours=0):
    """Model `ExternalTaskSensor` — one poll of the metadata database.

    Args:
        db: The metadata database.
        external_dag_id: The DAG being waited on.
        logical_date: The logical date being waited for.
        execution_delta_hours: How far apart the two runs' dates are.

    Returns:
        A dict with the poll result.
    """
    target = datetime.strptime(logical_date, "%Y-%m-%d").replace(
        tzinfo=timezone.utc)
    target = target - timedelta(hours=execution_delta_hours)
    target_date = target.date().isoformat()

    record = db.find_run(external_dag_id, target_date)
    return {
        "external_dag_id": external_dag_id,
        "looked_for": target_date,
        "found": record is not None,
        "state": record.state if record else None,
    }


def dataset_consumer_queue(db, *, uri, consumer_dag_id):
    """Model a Dataset consumer being queued by an update.

    Args:
        db: The metadata database.
        uri: The dataset URI.
        consumer_dag_id: The consuming DAG.

    Returns:
        A dict describing the queued run.
    """
    events = db.consumers_for(uri)
    if not events:
        return {"queued": False, "reason": "no dataset update recorded"}
    latest = events[-1]
    # A dataset-triggered run gets its OWN logical date — the time it was
    # queued — not the producer's. The producer's date travels in the
    # event metadata instead.
    run = db.create_run(
        consumer_dag_id,
        latest["producer_logical_date"],
        conf={"producer_logical_date": latest["producer_logical_date"]},
    )
    return {
        "queued": True,
        "consumer": consumer_dag_id,
        "run_id": run.run_id,
        "producer_logical_date": latest["producer_logical_date"],
        "event_extra": latest["extra"],
    }


# ---------------------------------------------------------------------
# Exercise the three mechanisms
# ---------------------------------------------------------------------
db = MetadataDb()

print("=" * 70)
print("1. TriggerDagRunOperator — push")
print("=" * 70)
result = trigger_dag_run(
    db,
    trigger_dag_id="orders_ingest",
    logical_date="2026-01-15",
    conf={"parent_run_id": "scheduled__2026-01-15", "source": "gcs"},
)
print(f"  triggered  : {result['triggered']}")
print(f"  run_id     : {result['run_id']}")
print(f"  conf stored: {result['conf']}")
print(f"  parent slot held: {result['parent_slot_held']}")
print()

# A second trigger for the same date fails without reset_dag_run.
try:
    trigger_dag_run(db, trigger_dag_id="orders_ingest",
                    logical_date="2026-01-15")
except ValueError as exc:
    print(f"  second trigger without reset_dag_run -> {type(exc).__name__}: {exc}")

reset = trigger_dag_run(db, trigger_dag_id="orders_ingest",
                        logical_date="2026-01-15", reset_dag_run=True)
print(f"  with reset_dag_run=True -> {reset['run_id']} (recreated)")
print()

# A non-serialisable conf is rejected at trigger time.
try:
    trigger_dag_run(db, trigger_dag_id="orders_ingest",
                    logical_date="2026-01-16",
                    conf={"when": datetime(2026, 1, 15)})
except TypeError as exc:
    print(f"  non-JSON conf -> {type(exc).__name__}: {exc}")
print()

print("=" * 70)
print("2. ExternalTaskSensor — pull")
print("=" * 70)
poll = external_task_sensor_poll(db, external_dag_id="orders_ingest",
                                 logical_date="2026-01-15")
print(f"  poll 1: {poll}")
# Mark it complete, then poll again.
db.runs[("orders_ingest", "2026-01-15")].state = "success"
poll = external_task_sensor_poll(db, external_dag_id="orders_ingest",
                                 logical_date="2026-01-15")
print(f"  poll 2: {poll}")
# A date that was never run: the sensor would keep polling until it timed out.
poll = external_task_sensor_poll(db, external_dag_id="orders_ingest",
                                 logical_date="2026-01-14")
print(f"  poll for a date that never ran: {poll}")
print("  -> the sensor would keep polling until its timeout fired.")
print()

print("=" * 70)
print("3. Dataset — declarative")
print("=" * 70)
event = db.record_dataset_update(
    ORDERS_DATASET_URI,
    producer_dag_id="orders_ingest",
    logical_date="2026-01-15",
    extra={"logical_date": "2026-01-15", "row_count": 1245},
)
print(f"  update recorded: {event['uri']}")
print(f"  producer date  : {event['producer_logical_date']}")
print(f"  event metadata : {event['extra']}")
print()

queued = dataset_consumer_queue(db, uri=ORDERS_DATASET_URI,
                                consumer_dag_id="consume_orders")
print(f"  consumer queued: {queued['queued']}")
print(f"  producer date carried through: {queued['producer_logical_date']}")
print(f"  event metadata available      : {queued['event_extra']}")
print()

# --- the gap: a dataset cannot time out -------------------------------
print("  Scenario: the producer never runs.")
no_update = db.consumers_for("gs://YOUR_BUCKET/datasets/never_updated")
print(f"  dataset updates recorded: {len(no_update)}")
print("  -> the consumer simply never runs, and nothing reports it.")
print("     A Dataset cannot time out. This needs a monitoring check.")

## 6. The "wait for data" pattern

### Why a sensor, and why not a sleep loop

A sensor **resumes** the task when its condition is met; it does not busy-wait in a loop. The Airflow worker is released while the sensor waits, and the task is re-evaluated later. That is the property that makes sensors the right tool for "wait for X" — and a `time.sleep` loop the wrong one.

A sleep loop holds a worker slot for the whole wait. On a small Composer environment, three tasks waiting an hour each can consume every slot and block the entire pipeline.

```python
sensor = PythonSensor(
    task_id="wait_for_upstream_data",
    python_callable=wait_for_upstream_data,
    mode="reschedule",       # release the slot while waiting
    timeout=3600,
    poke_interval=300,
    soft_fail=True,
)
```

| `mode` | Behaviour | Use for |
|---|---|---|
| `poke` | Holds the worker slot, re-checks every `poke_interval` | short waits (seconds to a minute) |
| `reschedule` | Releases the slot, the scheduler re-queues the task | long waits — **use this** |
| `deferrable` | The task exits entirely; a trigger resumes it | very long waits on Airflow 2.2+ |

### The three variants of "wait for data"

| Variant | Situation | Mechanism |
|---|---|---|
| **A** | The data arrives on a known schedule | Don't wait. Schedule the consumer after the producer's known completion time. Use an `ExternalTaskSensor` only if you need to *confirm* the producer ran. |
| **B** | The data arrives when it arrives, and **you own the producer** | Use a **Dataset**. No polling, no slot. |
| **C** | The data arrives when it arrives, and **you do not own the producer** | Use a **sensor**, in `reschedule` mode, with a bounded timeout and a deliberate `soft_fail` policy. |

```text
Do you know when the data arrives?
├── YES → schedule it. (Variant A)
└── NO
    └── Do you control the producer?
        ├── YES → Dataset. (Variant B)
        └── NO  → Sensor, reschedule mode, bounded timeout. (Variant C)
```

### The condition belongs in `core/`, not in the DAG

Sensors are orchestration. The *condition* they evaluate is logic, and logic belongs in a pure function where it can be unit-tested with no Airflow, no worker, and no timeout.

That is exactly what `project/core/sensor_conditions.py` is:

```python
def source_is_ready(
    lines: list[str],
    *,
    expected_rows: int = 0,
    reject_threshold: int = 0,
) -> bool:
    """Return True when the source data is complete enough to proceed."""
    accepted, rejected = transform_orders(lines)
    if len(accepted) < expected_rows:
        return False
    if len(rejected) > reject_threshold:
        return False
    return True
```

Keeping it pure means the "is the data ready?" rule is tested directly, rather than being rediscovered every time the sensor times out at 03:00.

In [ ]:
"""The 'wait for data' pattern, built on project/core/sensor_conditions.py.

The sensor's *condition* is pure and unit-tested. The sensor's *loop* is
Airflow's job: it re-evaluates the condition every `poke_interval` until
it returns True or the timeout fires.
"""

from local_runner import DAG, Task

#: The pure readiness condition.
from sensor_conditions import source_is_ready


def sensor_poll(run, context):
    """Simulate one poke of the sensor loop.

    A sensor does not block. It is re-evaluated on a schedule until the
    condition is true. This function performs one evaluation and records
    the outcome, so the loop can be inspected.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A dict describing the poll.
    """
    lines = run.pull("read_orders") or []
    expected_rows = int(context["params"].get("expected_rows", 0))
    reject_threshold = int(context["params"].get("reject_threshold", 0))
    ready = source_is_ready(lines, expected_rows=expected_rows,
                            reject_threshold=reject_threshold)
    attempts = int(run.pull("sensor_poll", "attempts") or 0) + 1
    run.xcom.push("sensor_poll", "attempts", attempts)
    return {"attempt": attempts, "ready": ready}


def process_when_ready(run, context):
    """The work that runs once the data is ready.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A description of the work done.
    """
    return "processed"


def build_wait_dag():
    """Build the wait-for-data DAG.

    Returns:
        The DAG.
    """
    return DAG(
        "patterns_wait_for_data",
        tasks=[
            Task("read_orders", read_orders),
            Task("sensor_poll", sensor_poll, dependencies=["read_orders"]),
            Task("process", process_when_ready, dependencies=["sensor_poll"]),
        ],
        schedule="@daily",
        start_date="2026-01-01",
        description="Wait for the source data, then process it.",
    )


WAIT_DAG = build_wait_dag()
print(WAIT_DAG.describe())
print()

# ---------------------------------------------------------------------
# The condition, exercised against real data
# ---------------------------------------------------------------------
from sensor_conditions import source_is_ready  # noqa: E402

CLEAN = (COURSE_ROOT / "sample_data" / "orders.csv").read_text(
    encoding="utf-8").splitlines()
MALFORMED = (COURSE_ROOT / "sample_data" / "malformed_orders.csv").read_text(
    encoding="utf-8").splitlines()

print("The pure condition, `source_is_ready`, against real data:")
print()

cases = [
    ("no expectations, clean data", CLEAN, 0, 0),
    ("expect 1000 rows, clean data", CLEAN, 1000, 0),
    ("expect 100000 rows, clean data", CLEAN, 100_000, 0),
    ("no expectations, malformed data", MALFORMED, 0, 0),
    ("expect 1000 rows, malformed data", MALFORMED, 1000, 0),
    ("tolerate 8 rejects, malformed data", MALFORMED, 0, 8),
]

for label, lines, expected, threshold in cases:
    ready = source_is_ready(lines, expected_rows=expected,
                            reject_threshold=threshold)
    print(f"  {label:<38} expected_rows={expected:<7} "
          f"reject_threshold={threshold:<3} -> ready={ready}")

print()
print("Read the table as the sensor's decision rule:")
print("  * too few accepted rows  -> not ready")
print("  * too many rejected rows -> not ready")
print("  * otherwise              -> ready")
print()

# The three cases the sensor's timeout would fire on.
assert source_is_ready(CLEAN, expected_rows=1000, reject_threshold=0) is True
assert source_is_ready(CLEAN, expected_rows=100_000, reject_threshold=0) is False
assert source_is_ready(MALFORMED, expected_rows=0, reject_threshold=0) is False
assert source_is_ready(MALFORMED, expected_rows=0, reject_threshold=8) is True

print("PASS: the condition behaves as documented.")
print()

# ---------------------------------------------------------------------
# Run the wait-for-data DAG
# ---------------------------------------------------------------------
from local_runner import run_dag  # noqa: E402

print("=" * 70)
print("WAIT FOR DATA: the sensor fires immediately")
print("=" * 70)
run = run_dag(WAIT_DAG, "2026-01-15",
              params={"expected_rows": 1000, "reject_threshold": 0},
              verbose=True)
assert run.state["sensor_poll"] == "success", run.state
assert run.state["process"] == "success", run.state
print("PASS: the sensor succeeded and the work ran.")
print()

print("=" * 70)
print("WAIT FOR DATA: the sensor never fires")
print("=" * 70)
timeout_run = run_dag(WAIT_DAG, "2026-01-15",
                      params={"expected_rows": 100_000, "reject_threshold": 0},
                      verbose=True)
poll = timeout_run.pull("sensor_poll")
print(f"poll outcome: {poll}")
print()
print("The condition returned False — the data was not ready.")
print()
print("In Airflow the difference is what happens NEXT: a `PythonSensor`")
print("re-raises the poke, so the task stays in the running state and is")
print("re-evaluated every `poke_interval` until the timeout expires, at")
print("which point it fails with:")
print("    airflow.exceptions.SensorTimeout: ...")
print("The mini-runner has no poke loop, so it simply records the False")
print("and finishes. The important part is the DECISION: the condition")
print("said 'not ready', and the pipeline had no business proceeding.")
print("Better a failed run than a run that processed half a file.")

## 7. Idempotency — the property every pattern needs

A pattern is only safe if running it twice produces the same result as running it once. That sounds obvious and is the single most commonly violated property in production pipelines.

### The two mechanisms

**Idempotent write** — the destination is replaced, not appended to. `idempotent_load.apply_load` implements both modes:

```python
apply_load(existing, incoming, mode="truncate", partitions=None)
apply_load(existing, incoming, mode="append")
```

- `truncate` replaces the rows in scope (all incoming regions, or just the named `partitions`). A retry writes the same partition again — same result.
- `append` adds to what is there. A retry **doubles** the numbers. This is the `WRITE_APPEND` bug.

**Deduplication** — the destination is append-only, but each row's identity is derived from the data, so a duplicate is recognisable. `dedup_load.dedup_rows` keeps the last row for each business key:

```python
dedup_rows(rows, partition)
```

The identity comes from `pipeline_logic.dedup_key`:

```python
def dedup_key(order_id: str, partition: str) -> str:
    return f"{partition}:{order_id}"
```

> **The identity must be derived from the data, never from arrival time.** A load-time timestamp would make a retry produce *different* rows rather than the same rows, which defeats deduplication entirely.

### The three failure modes they prevent

| Failure | Without idempotency | With it |
|---|---|---|
| The task is retried after a partial write | Rows counted twice | The partition is replaced |
| The DAG is re-run for the same date | The day's revenue doubles | The partition is replaced |
| A downstream job re-reads a partition | It sees a different row count each time | It sees the same row count every time |

That third one is the subtle one. **Idempotency is not only about the destination — it is about the read.** A pipeline that is not idempotent produces a different answer every time you ask, and every downstream consumer then has to be defensive about it.

In [ ]:
"""Idempotency: the same pipeline, run twice, must produce the same answer."""

from dedup_load import dedup_rows
from idempotent_load import apply_load
from pipeline_logic import (
    build_bigquery_rows,
    dedup_key,
    filter_rows_for_partition,
    normalise_currency,
    partition_for_date,
    summarise_by_region,
    transform_orders,
)

lines = (COURSE_ROOT / "sample_data" / "orders.csv").read_text(
    encoding="utf-8").splitlines()
accepted, _ = transform_orders(lines)
summary = summarise_by_region(accepted)

PARTITION = partition_for_date("2026-01-15")
print(f"partition for 2026-01-15: {PARTITION}")
print()

# ---------------------------------------------------------------------
# 1. Truncate: a retry replaces the partition
# ---------------------------------------------------------------------
print("=" * 70)
print("1. TRUNCATE — the idempotent write")
print("=" * 70)
empty = {}

first = apply_load(empty, summary, mode="truncate")
second = apply_load(first, summary, mode="truncate")

print(f"  after run 1: {first}")
print(f"  after run 2: {second}")
assert first == second, "a truncate write must be idempotent"
assert first == summary, "a truncate write must reproduce the source"
print("PASS: run 2 produced exactly the same table as run 1.")
print()

# A *partial* scope: only one partition is replaced.
scoped = apply_load(first, summary, mode="truncate", partitions=["AMER"])
assert scoped["AMER"] == summary["AMER"]
assert scoped["APAC"] == summary["APAC"]
print("PASS: a partition-scoped truncate leaves the other regions alone.")
print()

# ---------------------------------------------------------------------
# 2. Append: the same input, counted twice
# ---------------------------------------------------------------------
print("=" * 70)
print("2. APPEND — the bug this pattern exists to prevent")
print("=" * 70)
appended = apply_load(first, summary, mode="append")
appended_again = apply_load(appended, summary, mode="append")

print(f"  revenue after 1 append : {appended['AMER']['revenue_eur']}")
print(f"  revenue after 2 appends: {appended_again['AMER']['revenue_eur']}")
assert appended_again["AMER"]["revenue_eur"] == round(
    summary["AMER"]["revenue_eur"] * 3, 2)
print("PASS: appending twice produced three times the revenue.")
print("      This is exactly what WRITE_APPEND does on a retry.")
print()

# ---------------------------------------------------------------------
# 3. Dedup: an append-only destination that survives a retry
# ---------------------------------------------------------------------
print("=" * 70)
print("3. DEDUP — idempotency without truncation")
print("=" * 70)
bq_rows = build_bigquery_rows(accepted)

# Simulate a retry: the same rows arrive again, plus one corrected row
# with the same business key (a late fix from the source system).
retry_rows = list(bq_rows) + [
    {**bq_rows[0], "amount": bq_rows[0]["amount"] + 100.0},
]

deduped = dedup_rows(retry_rows, PARTITION)
print(f"  rows in                : {len(retry_rows)}")
print(f"  rows after dedup       : {len(deduped)}")
print(f"  business key of row 0  : {dedup_key(bq_rows[0]['order_id'], PARTITION)}")
print(f"  amount before dedup    : {bq_rows[0]['amount']}")
print(f"  amount after dedup     : {deduped[0]['amount']}")

assert len(deduped) == len(bq_rows), "dedup must remove exactly the duplicate"
assert deduped[0]["amount"] == bq_rows[0]["amount"] + 100.0, \
    "the LAST row for a key must win"
print("PASS: the duplicate was removed and the later row won.")
print()

# Dedup is itself idempotent.
again = dedup_rows(deduped, PARTITION)
assert again == deduped, "dedup must be idempotent"
print("PASS: dedup is idempotent — running it again changes nothing.")
print()

# ---------------------------------------------------------------------
# 4. Partition scoping: the retry writes the SAME partition
# ---------------------------------------------------------------------
print("=" * 70)
print("4. PARTITION SCOPING — why the partition is part of the key")
print("=" * 70)
in_partition = filter_rows_for_partition(accepted, PARTITION)
print(f"  rows in partition {PARTITION}: {len(in_partition)} of {len(accepted)}")

# The same order_id in two different partitions is two different rows —
# which is why the key includes the partition.
key_a = dedup_key("o00000001", "20260115")
key_b = dedup_key("o00000001", "20260116")
print(f"  key in 20260115: {key_a}")
print(f"  key in 20260116: {key_b}")
assert key_a != key_b, "the partition must be part of the identity"

# But within one partition, the same order_id is the same row.
assert dedup_key("o00000001", "20260115") == key_a
print("PASS: identity is (partition, order_id) — stable, and data-derived.")
print()

# ---------------------------------------------------------------------
# 5. The end-to-end property
# ---------------------------------------------------------------------
print("=" * 70)
print("5. THE PROPERTY: run the pipeline twice, get the same answer")
print("=" * 70)


def run_pipeline_once():
    """Run the whole load and return the final table.

    Returns:
        The final table, region -> bucket.
    """
    rows, _ = transform_orders(lines)
    region_summary = summarise_by_region(rows)
    partition_rows = filter_rows_for_partition(rows, PARTITION)
    shaped = build_bigquery_rows(partition_rows)
    unique = dedup_rows(shaped, PARTITION)
    return apply_load({}, summarise_by_region(unique), mode="truncate")


run_1 = run_pipeline_once()
run_2 = run_pipeline_once()
assert run_1 == run_2, "the pipeline must be idempotent"
print(f"  run 1 == run 2 : {run_1 == run_2}")
for region in sorted(run_1):
    bucket = run_1[region]
    print(f"  {region:<6} order_count={int(bucket['order_count']):<5} "
          f"revenue_eur={bucket['revenue_eur']:.2f}")
print()
print("PASS: the pipeline is idempotent. A retry is a no-op.")
print()

# The amount conversion is also deterministic, which the whole property
# depends on.
print(f"  normalise_currency(100, 'GBP') = {normalise_currency(100, 'GBP')}")
print(f"  normalise_currency(100, 'EUR') = {normalise_currency(100, 'EUR')}")
print(f"  normalise_currency(100, 'JPY') = {normalise_currency(100, 'JPY')}")
print()
print("PASS: the whole chain — parse, convert, partition, dedup, load —")
print("      is a pure function of its input.")

## 8. Case study A — a daily orders ETL

The shape almost every production pipeline has.

```text
  read ──▶ validate ──▶ transform ──▶ load ──▶ aggregate ──▶ publish
```

**Patterns used:** linear pipeline, XCom between stages, a dead-letter path, an SLA.

**The decisions that matter:**

| Decision | Choice | Why |
|---|---|---|
| Where does the logic live? | `project/core/pipeline_logic.py` | Testable without Airflow |
| What crosses task boundaries? | A *reference*, not the data | XCom is a database; 1,245 rows in an XCom is a mistake |
| What happens to bad rows? | A dead-letter path with a reason code | Silently dropping them loses the incident |
| How is the load made safe to retry? | `WRITE_TRUNCATE` against a partition decorator | A retry replaces, not appends |
| How is the run kept honest? | `sla` + `sla_miss_callback` | The SLA is measured from the interval end, not the wall clock |

### The half-open interval — the off-by-one that hides for a month

Airflow's data interval for the `2026-01-15` run is `[2026-01-15, 2026-01-16)`. The SQL filter must match exactly:

```sql
WHERE order_ts >= '{{ data_interval_start }}'
  AND order_ts <  '{{ data_interval_end }}'
```

A filter written as `<= '2026-01-15'` double-counts the midnight row. One written with `BETWEEN` loads the boundary row into two days. This is the single most common bug in a daily Airflow pipeline, and it is invisible until the monthly totals are wrong.

Let us verify the interval arithmetic directly.

In [ ]:
"""Case study A: the daily orders ETL, end to end in the mini-runner."""

from local_runner import DAG, Task, run_dag
from pipeline_logic import (
    logical_date_window,
    partition_for_date,
    render_summary_csv,
)

LOGICAL_DATE = "2026-01-15"


def read_orders(run, context):
    """Read the raw orders.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The raw CSV lines.
    """
    path = COURSE_ROOT / "sample_data" / "orders.csv"
    return path.read_text(encoding="utf-8").splitlines()


def validate_and_stage(run, context):
    """Validate the rows and stage them, pushing a reference downstream.

    Note what crosses the task boundary: the *count*, not the rows. In
    Airflow the staged data lives in BigQuery and the task pushes the
    table name; here the staged rows live in XCom because they are small
    enough to be a teaching example. The shape is the same.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A dict describing the staging.
    """
    from pipeline_logic import summarise_by_region, transform_orders

    raw = run.pull("read_orders") or []
    accepted, rejected = transform_orders(raw)
    run.xcom.push("validate_and_stage", "accepted", accepted)
    run.xcom.push("validate_and_stage", "rejected", rejected)
    run.xcom.push("validate_and_stage", "summary", summarise_by_region(accepted))
    partition = partition_for_date(context["ds"])
    return {
        "accepted_count": len(accepted),
        "rejected_count": len(rejected),
        "staging_table": "analytics.orders_enriched_staging",
        "partition": partition,
        "partition_decorator": f"analytics.orders_enriched${partition}",
    }


def load_to_bigquery(run, context):
    """Load the staged partition, idempotently.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A description of the load.
    """
    staging = run.pull("validate_and_stage") or {}
    return (
        f"loaded {staging.get('accepted_count', 0)} rows into "
        f"{staging.get('partition_decorator', '?')} with WRITE_TRUNCATE"
    )


def aggregate_by_region(run, context):
    """Aggregate the loaded rows.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The region summary.
    """
    summary = run.pull("validate_and_stage", "summary") or {}
    run.xcom.push("aggregate_by_region", "summary", summary)
    return summary


def publish_summary(run, context):
    """Publish the summary to a GCS-shaped path.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The published URI.
    """
    summary = run.pull("aggregate_by_region", "summary") or {}
    bucket = context["__variables__"].get("target_bucket", "demo-target-bucket")
    uri = f"gs://{bucket}/{{dag.dag_id}}/{{ds}}/summary.csv"
    return {
        "uri": f"gs://{bucket}/etl_orders_to_bigquery/{context['ds']}/summary.csv",
        "template": uri,
        "csv": render_summary_csv(summary),
    }


def build_daily_etl_dag():
    """Build the daily ETL DAG.

    Returns:
        The DAG.
    """
    return DAG(
        "case_study_daily_etl",
        tasks=[
            Task("read_orders", read_orders),
            Task("validate_and_stage", validate_and_stage,
                 dependencies=["read_orders"]),
            Task("load_to_bigquery", load_to_bigquery,
                 dependencies=["validate_and_stage"]),
            Task("aggregate_by_region", aggregate_by_region,
                 dependencies=["load_to_bigquery"]),
            Task("publish_summary", publish_summary,
                 dependencies=["aggregate_by_region"]),
        ],
        schedule="@daily",
        start_date="2026-01-01",
        description="Read, validate, load, aggregate, publish.",
    )


DAILY_ETL_DAG = build_daily_etl_dag()
print(DAILY_ETL_DAG.describe())
print()

# ---------------------------------------------------------------------
# The half-open interval, verified
# ---------------------------------------------------------------------
print("=" * 70)
print("The half-open data interval")
print("=" * 70)
start, end = logical_date_window(LOGICAL_DATE)
print(f"  logical date       : {LOGICAL_DATE}")
print(f"  data_interval_start: {start}")
print(f"  data_interval_end  : {end}   (exclusive)")
print(f"  partition decorator: {partition_for_date(LOGICAL_DATE)}")
print()
print("  The SQL filter must be:")
print(f"    WHERE order_ts >= '{start}'")
print(f"      AND order_ts <  '{end}'")
print()
print("  A filter using '<=' would double-count the row at exactly midnight.")
print()

run = run_dag(DAILY_ETL_DAG, LOGICAL_DATE, verbose=True)
print()
print("=" * 70)
print("The result")
print("=" * 70)
published = run.pull("publish_summary")
print(f"  published to : {published['uri']}")
print()
print(published["csv"], end="")

EXPECTED = {
    "AMER": {"order_count": 318, "revenue_eur": 113592.19},
    "APAC": {"order_count": 285, "revenue_eur": 108194.92},
    "EMEA": {"order_count": 322, "revenue_eur": 118423.22},
    "LATAM": {"order_count": 320, "revenue_eur": 110646.18},
}
staging = run.pull("validate_and_stage")
summary = run.pull("aggregate_by_region", "summary")
assert summary == EXPECTED, f"summary changed: {summary}"
assert staging["accepted_count"] == 1245, staging
assert staging["rejected_count"] == 0, staging
print("PASS: the daily ETL produces the expected summary, "
      "accepted=1245 rejected=0.")

## 9. Case study B — a backfill

A backfill re-runs a pipeline for a range of dates that have already passed. It is the pattern most likely to go wrong, because it runs against a table that already has data.

### The three things a backfill must get right

| # | Requirement | Why |
|---|---|---|
| 1 | **One DAG run per date** | Airflow's unit of work is a run, and a run has one logical date. A backfill of 7 days is 7 runs. |
| 2 | **`depends_on_past=False`** | The default is `True` for catchup runs. With `True`, the 15th waits for the 14th — which serialises the backfill and means one failure blocks every later date. |
| 3 | **Idempotent writes** | The destination already has rows for that date. The write must replace, not append. |

### Triggering a backfill

Three ways, in increasing order of control:

```bash
# 1. The UI — click the date on the Grid view, then "Trigger DAG with config".
# 2. The CLI:
gcloud composer environments run ... --dag etl_orders_to_bigquery \
    --run-date 2026-01-10

# 3. Programmatically, for a range of dates:
```

```python
from airflow.api.client.local_client import Client
from airflow.utils.types import DagRunArgType

client = Client()
for day in ["2026-01-08", "2026-01-09", "2026-01-10"]:
    client.trigger_dag(
        dag_id="etl_orders_to_bigquery",
        run_id=f"backfill__{day}",
        execution_date=day,
        conf={"backfill": True},
    )
```

### The `max_active_runs` interaction

A backfill with `max_active_runs=1` (the safe default for a pipeline that writes the same table) runs **one date at a time**. With `max_active_runs=5`, five dates run concurrently — which is faster, and is exactly how you get two dates writing the same partition if the partition key is wrong.

### `catchup` is not a backfill

`catchup=True` on a DAG with a `start_date` in the past makes the scheduler create a run for **every** missed interval since that date. It is not a backfill — it is an accident waiting to happen, and it can start a thousand runs at once. Leave `catchup=False` and trigger backfills explicitly.

Let us build a backfill runner and prove the property that matters: **a backfilled date produces the same result as an original run.**

In [ ]:
"""Case study B: a backfill, and the property that makes it safe."""

from local_runner import run_dag
from pipeline_logic import partition_for_date, render_summary_csv

LOGICAL_DATE = "2026-01-15"


def read_orders(run, context):
    """Read the raw orders for this run's date.

    In a real backfill this reads the *specific* date's file, not
    "whatever is in the bucket now". Here the sample dataset is a single
    day, so every date reads the same source — and the point of the
    exercise is the *destination* behaviour, not the source.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The raw CSV lines.
    """
    path = COURSE_ROOT / "sample_data" / "orders.csv"
    return path.read_text(encoding="utf-8").splitlines()


def validate_and_stage(run, context):
    """Validate the rows and stage them for the run's partition.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A dict describing the staging.
    """
    from pipeline_logic import summarise_by_region, transform_orders

    raw = run.pull("read_orders") or []
    accepted, rejected = transform_orders(raw)
    run.xcom.push("validate_and_stage", "accepted", accepted)
    run.xcom.push("validate_and_stage", "summary", summarise_by_region(accepted))
    return {
        "accepted_count": len(accepted),
        "rejected_count": len(rejected),
        "partition": partition_for_date(context["ds"]),
    }


def load_to_bigquery(run, context):
    """Load the staged partition, replacing what is already there.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A description of the load.
    """
    staging = run.pull("validate_and_stage") or {}
    return (
        f"WRITE_TRUNCATE into {staging['partition']} "
        f"({staging['accepted_count']} rows)"
    )


def aggregate_by_region(run, context):
    """Aggregate the loaded rows.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The region summary.
    """
    summary = run.pull("validate_and_stage", "summary") or {}
    run.xcom.push("aggregate_by_region", "summary", summary)
    return summary


from local_runner import DAG, Task  # noqa: E402

BACKFILL_DAG = DAG(
    "case_study_backfill",
    tasks=[
        Task("read_orders", read_orders),
        Task("validate_and_stage", validate_and_stage, dependencies=["read_orders"]),
        Task("load_to_bigquery", load_to_bigquery, dependencies=["validate_and_stage"]),
        Task("aggregate_by_region", aggregate_by_region,
             dependencies=["load_to_bigquery"]),
    ],
    # catchup=False: a backfill is triggered explicitly, one date at a time.
    schedule="@daily",
    start_date="2026-01-01",
    description="One run per date; the write replaces the partition.",
)

print(BACKFILL_DAG.describe())
print()
print(f"catchup          : {BACKFILL_DAG.catchup}")
print(f"max_active_runs  : {BACKFILL_DAG.max_active_runs}")
print()

# ---------------------------------------------------------------------
# A backfill: one run per date
# ---------------------------------------------------------------------
BACKFILL_DATES = ["2026-01-08", "2026-01-09", "2026-01-10", "2026-01-11"]

print("=" * 70)
print("THE BACKFILL: one DAG run per date")
print("=" * 70)

results = {}
for day in BACKFILL_DATES:
    run = run_dag(BACKFILL_DAG, day)
    assert run.state["load_to_bigquery"] == "success", run.state
    partition = run.pull("validate_and_stage")["partition"]
    results[day] = run.pull("aggregate_by_region", "summary")
    print(f"  {day}  partition={partition}  "
          f"rows={run.pull('validate_and_stage')['accepted_count']}  "
          f"state={'ok' if all(s == 'success' for s in run.state.values()) else 'FAILED'}")

print()
print("PASS: every backfilled date produced a successful run.")
print()

# ---------------------------------------------------------------------
# The property: a backfilled date gives the same answer as an original run
# ---------------------------------------------------------------------
print("=" * 70)
print("THE PROPERTY: backfill == original run")
print("=" * 70)

original = run_dag(BACKFILL_DAG, LOGICAL_DATE)
original_summary = original.pull("aggregate_by_region", "summary")
backfilled_summary = results[BACKFILL_DATES[0]]

assert original_summary == backfilled_summary, (
    f"backfill changed the answer: {original_summary} != {backfilled_summary}"
)
print("PASS: the backfilled run produced exactly the same table as the")
print("      original run. That is what makes a backfill safe.")
print()

# ---------------------------------------------------------------------
# The failure a non-idempotent backfill produces
# ---------------------------------------------------------------------
print("=" * 70)
print("THE FAILURE: what an append-mode backfill does")
print("=" * 70)
from idempotent_load import apply_load  # noqa: E402

table = {}
table = apply_load(table, original_summary, mode="append")   # original run
table = apply_load(table, backfilled_summary, mode="append")  # backfill
table = apply_load(table, backfilled_summary, mode="append")  # retry

print(f"  revenue after one original + two backfills: "
      f"{table['AMER']['revenue_eur']:.2f}")
print(f"  correct revenue                            : "
      f"{original_summary['AMER']['revenue_eur']:.2f}")
print()
print("  The backfill ran twice, so the revenue is three times the truth.")
print("  The numbers look plausible — which is exactly why this bug")
print("  survives for weeks before anyone notices.")
print()

# The correct behaviour, for contrast.
safe = apply_load({}, original_summary, mode="truncate")
safe = apply_load(safe, backfilled_summary, mode="truncate")
safe = apply_load(safe, backfilled_summary, mode="truncate")
assert safe == original_summary
print("  With WRITE_TRUNCATE, the same three writes leave the table")
print("  exactly as the original run left it:")
print()
print(render_summary_csv(safe), end="")
print("PASS: truncate is idempotent; append is not.")

## 10. Case study C — a multi-DAG dependency chain

Three DAGs, each with a clear artefact boundary.

```text
  dag_ingest ──▶ dag_transform ──▶ dag_publish
       │                │                 │
       └─ writes ───────┴─ writes ────────┘
         raw.orders       analytics.orders  gs://.../summary.csv
```

### Where to split

> **Split by *artefact boundary*, not by *phase boundary*.**

A good boundary is a point where one DAG's output is a well-defined artefact another DAG can depend on. A bad boundary is "after the extract step" — because "the extract" is not an artefact, it is a phase. If you split there, you have two DAGs that both need to know about the extract's internals, and neither can be re-run alone.

The chain above splits at `raw.orders` and at `analytics.orders_enriched`. Both are tables. Both are queryable. Both are stable.

### Three ways to wire it, and what each costs

| Mechanism | Coupling | Failure visibility | Use when |
|---|---|---|---|
| `TriggerDagRunOperator` | The child knows the parent's name | The parent's trigger task fails | You own both DAGs — **the default choice** |
| `ExternalTaskSensor` | The child knows the parent's name **and task ids** | The sensor times out | You cannot restructure |
| **Dataset** | Nobody knows anybody's name | The consumer never runs | The coupling is on *data*, not control flow |

### The `conf` contract

The chain passes a small amount of context down:

```python
trigger_next = TriggerDagRunOperator(
    task_id="trigger_downstream_dag",
    trigger_dag_id="transform_orders_dag",
    wait_for_completion=False,
    conf={
        "source_partition": "{{ ds }}",
        "parent_run_id": "{{ run_id }}",
    },
    reset_dag_run=True,       # a backfill may re-trigger the same date
)
```

**Rules for a `conf` contract:**

1. **It must be JSON-serialisable.** A `datetime` raises `TypeError`.
2. **It must be small.** It is stored in the metadata database, one copy per run.
3. **It must be versioned in practice.** If the child changes what it expects, bump a `conf_version` key rather than silently changing the shape.
4. **`reset_dag_run=True` for anything a backfill touches.** Without it, the second trigger for a date fails with a duplicate-run error.

Let us build the chain and run it.

In [ ]:
"""Case study C: a three-DAG dependency chain, wired with a push trigger."""

import json
from dataclasses import dataclass, field
from typing import Any

from local_runner import DAG, Task, run_dag
from pipeline_logic import render_summary_csv, summarise_by_region, transform_orders

LOGICAL_DATE = "2026-01-15"


@dataclass
class ChainState:
    """The shared state the chain's DAGs read and write.

    In Airflow this state lives in three places: the metadata database
    (run states), XCom (small control data), and the warehouse (the
    artefacts). Here we model it with one object so the chain is
    readable.

    Attributes:
        artefacts: The artefacts each DAG has produced, by DAG id.
        runs: The run records, by DAG id.
    """

    artefacts: dict[str, Any] = field(default_factory=dict)
    runs: dict[str, str] = field(default_factory=dict)

    def record(self, dag_id, artefact):
        """Record a DAG's artefact.

        Args:
            dag_id: The DAG that produced it.
            artefact: The artefact.
        """
        self.artefacts[dag_id] = artefact
        self.runs[dag_id] = "success"


#: The chain's shared state.
CHAIN = ChainState()


# ---------------------------------------------------------------------
# DAG 1: ingest
# ---------------------------------------------------------------------
def ingest_read(run, context):
    """Read the raw orders.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The raw CSV lines.
    """
    path = COURSE_ROOT / "sample_data" / "orders.csv"
    return path.read_text(encoding="utf-8").splitlines()


def ingest_write_raw(run, context):
    """Write the raw orders to the raw table.

    The artefact boundary: `raw.orders` is a table another DAG can read.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The artefact reference.
    """
    raw = run.pull("ingest_read") or []
    table = "raw.orders"
    CHAIN.record("dag_ingest", {"table": table, "row_count": len(raw)})
    print(f"      wrote {len(raw)} rows to {table}")
    return {"table": table, "row_count": len(raw)}


# ---------------------------------------------------------------------
# DAG 2: transform
# ---------------------------------------------------------------------
def transform_read_raw(run, context):
    """Read the raw table produced by the ingest DAG.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The raw CSV lines.
    """
    path = COURSE_ROOT / "sample_data" / "orders.csv"
    return path.read_text(encoding="utf-8").splitlines()


def transform_validate(run, context):
    """Validate and transform the raw rows.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        A dict with the accepted and rejected counts.
    """
    raw = run.pull("transform_read_raw") or []
    accepted, rejected = transform_orders(raw)
    run.xcom.push("transform_validate", "accepted", accepted)
    run.xcom.push("transform_validate", "summary", summarise_by_region(accepted))
    return {"accepted_count": len(accepted), "rejected_count": len(rejected)}


def transform_write_curated(run, context):
    """Write the curated table — the second artefact boundary.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The artefact reference.
    """
    stats = run.pull("transform_validate") or {}
    table = "analytics.orders_enriched"
    partition = context["ds_nodash"]
    CHAIN.record("dag_transform", {"table": table, "partition": partition,
                                   "row_count": stats.get("accepted_count", 0)})
    print(f"      wrote {stats.get('accepted_count', 0)} rows to "
          f"{table}${partition}")
    return {"table": table, "partition": partition,
            "row_count": stats.get("accepted_count", 0)}


# ---------------------------------------------------------------------
# DAG 3: publish
# ---------------------------------------------------------------------
def publish_read_curated(run, context):
    """Read the curated table.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The accepted rows.
    """
    raw = (COURSE_ROOT / "sample_data" / "orders.csv").read_text(
        encoding="utf-8").splitlines()
    accepted, _ = transform_orders(raw)
    return accepted


def publish_write_summary(run, context):
    """Write the summary to GCS.

    Args:
        run: The run context.
        context: The Jinja context.

    Returns:
        The artefact reference.
    """
    accepted = run.pull("publish_read_curated") or []
    summary = summarise_by_region(accepted)
    bucket = context["__variables__"].get("target_bucket", "demo-target-bucket")
    uri = f"gs://{bucket}/summary/{context['ds']}/summary.csv"
    CHAIN.record("dag_publish", {"uri": uri, "summary": summary})
    print(f"      wrote {uri}")
    return {"uri": uri, "summary": summary}


# ---------------------------------------------------------------------
# The DAGs
# ---------------------------------------------------------------------
INGEST_DAG = DAG(
    "dag_ingest",
    tasks=[
        Task("ingest_read", ingest_read),
        Task("ingest_write_raw", ingest_write_raw, dependencies=["ingest_read"]),
    ],
    schedule="@daily",
    start_date="2026-01-01",
    description="Land the raw orders in raw.orders.",
)

TRANSFORM_DAG = DAG(
    "dag_transform",
    tasks=[
        Task("transform_read_raw", transform_read_raw),
        Task("transform_validate", transform_validate,
             dependencies=["transform_read_raw"]),
        Task("transform_write_curated", transform_write_curated,
             dependencies=["transform_validate"]),
    ],
    schedule=None,          # triggered by the ingest DAG
    start_date="2026-01-01",
    description="Validate and curate raw.orders into analytics.orders_enriched.",
)

PUBLISH_DAG = DAG(
    "dag_publish",
    tasks=[
        Task("publish_read_curated", publish_read_curated),
        Task("publish_write_summary", publish_write_summary,
             dependencies=["publish_read_curated"]),
    ],
    schedule=None,          # triggered by the transform DAG
    start_date="2026-01-01",
    description="Publish the per-region summary to Cloud Storage.",
)

print("The chain:")
print()
for dag in (INGEST_DAG, TRANSFORM_DAG, PUBLISH_DAG):
    print(f"  {dag.dag_id:<16} schedule={dag.schedule!r:<10} "
          f"tasks={len(dag.tasks)}")
print()
print("  dag_ingest --raw.orders--> dag_transform "
      "--analytics.orders_enriched--> dag_publish")
print()

# ---------------------------------------------------------------------
# Run the chain
# ---------------------------------------------------------------------
print("=" * 70)
print("RUNNING THE CHAIN")
print("=" * 70)

for dag in (INGEST_DAG, TRANSFORM_DAG, PUBLISH_DAG):
    run = run_dag(dag, LOGICAL_DATE, verbose=True)
    failed = [t for t, s in run.state.items() if s != "success"]
    assert not failed, f"{dag.dag_id} failed: {failed}"
    print()

print("Artefacts produced:")
for dag_id, artefact in CHAIN.artefacts.items():
    printable = {k: v for k, v in artefact.items() if k != "summary"}
    print(f"  {dag_id:<16} {printable}")
print()

published = CHAIN.artefacts["dag_publish"]
print("The published summary:")
print()
print(render_summary_csv(published["summary"]), end="")

EXPECTED = {
    "AMER": {"order_count": 318, "revenue_eur": 113592.19},
    "APAC": {"order_count": 285, "revenue_eur": 108194.92},
    "EMEA": {"order_count": 322, "revenue_eur": 118423.22},
    "LATAM": {"order_count": 320, "revenue_eur": 110646.18},
}
assert published["summary"] == EXPECTED, published["summary"]
print("PASS: the three-DAG chain produces the expected summary.")
print()

# ---------------------------------------------------------------------
# The conf contract
# ---------------------------------------------------------------------
print("=" * 70)
print("THE CONF CONTRACT")
print("=" * 70)
conf = {
    "conf_version": 1,
    "source_partition": "20260115",
    "parent_run_id": "scheduled__2026-01-15T00:00:00+00:00",
}
serialised = json.dumps(conf, sort_keys=True)
print(f"  conf as stored in the metadata DB: {serialised}")
print(f"  size: {len(serialised)} bytes")
print()
print("  Rules:")
print("   1. JSON-serialisable — a datetime raises TypeError.")
print("   2. Small — one copy per run, in the metadata database.")
print("   3. Versioned — bump conf_version rather than changing the shape.")
print("   4. reset_dag_run=True for anything a backfill touches.")
print()

try:
    from datetime import datetime
    json.dumps({"when": datetime(2026, 1, 15)})
except TypeError as exc:
    print(f"  a datetime in the conf -> TypeError: {exc}")

## 11. The anti-patterns

Five patterns that look reasonable and are not.

### 1. The task-generating loop

Covered in section 3. The width is fixed at parse time, every task exists in every run, and a loop over hundreds of items slows the scheduler's parsing of *every* DAG in the environment.

**A loop is still acceptable** when the number of items is genuinely static and small — say, three known environments — and you want each to have its own SLA. Just never use a loop to express "however many rows there are".

### 2. `SubDagOperator`

Removed in Airflow 2.0. It existed to nest a DAG inside a DAG, and it was confusing enough that Airflow deleted it. Use `TaskGroup`:

```python
from airflow.utils.task_group import TaskGroup

with TaskGroup(group_id="load") as load_group:
    load_amer = PythonOperator(task_id="load_amer", ...)
    load_apac = PythonOperator(task_id="load_apac", ...)
```

A `TaskGroup` is a **UI grouping**, not a scheduling unit. Its tasks are addressed as `load.load_amer`. It does not create a scheduling boundary — tasks in a group still run independently.

### 3. The one giant DAG

A DAG with 200 tasks that does everything — extract from four sources, transform, load to three destinations, reconcile, report. Problems:

- One failure fails everything downstream.
- The DAG graph is unreadable.
- Retries are all-or-nothing in terms of what re-runs.
- There is no way to run the reconciliation without the extract.

**Split it by artefact boundary**, as case study C does.

### 4. Using Airflow as a data processing engine

The rule from the anti-pattern chapter, worth restating:

> **If the task takes more than a few minutes, or holds more than a few hundred megabytes, it belongs in a processing engine.**

A `PythonOperator` is a single process: no parallelism, no autoscaling, a retry that starts from zero, no checkpointing, and untestable logic.

### 5. Sensors as a substitute for good contracts

A sensor that waits for a condition that will *never* be true is a timeout waiting to happen. If the contract says "the file appears by 02:00", schedule the consumer at 03:00 and use the sensor only to confirm.

### A sixth, specific to fan-out: the join with the default trigger rule

We already built this one and watched it fail. It is worth naming separately because it is the most common way a fan-out pattern fails in production:

```python
# WRONG — the join never runs after a failed branch.
join = PythonOperator(task_id="join", python_callable=join_parts)

# RIGHT — the join runs with the partial results.
join = PythonOperator(
    task_id="join",
    python_callable=join_parts,
    trigger_rule=TriggerRule.NONE_FAILED_MIN_ONE_SUCCESS,
)
```

---

## 12. Choosing a trigger rule

Six scenarios, six answers. This table is worth memorising.

| Scenario | Trigger rule | Why |
|---|---|---|
| (a) A join after a `BranchOperator` where either branch may be taken | `NONE_FAILED_MIN_ONE_SUCCESS` | `all_success` never runs it after a failure; `none_failed` runs it even when everything was skipped |
| (b) A cleanup task that must run even if the load failed | `all_done` | "run regardless of what happened upstream" |
| (c) A join where every upstream must have succeeded and you want to fail loudly | `all_success` | the default — and the point |
| (d) A join where at least one upstream must have succeeded | `NONE_FAILED_MIN_ONE_SUCCESS` | ditto |
| (e) An alert task that should fire only if something failed | `one_failed` | fires on failure, not on success |
| (f) A task that must run after every upstream has finished, successful or not | `all_done` | same as (b) |

Note that (b) and (f) are the same rule, and (a) and (d) are the same rule. That is not a coincidence — the six scenarios reduce to four distinct rules, and you only ever need four.

## Exercises

1. **Pick the trigger rule.** For each of these joins, state the `trigger_rule` and justify it in one sentence:
   (a) a join after a `BranchOperator` where either branch may be taken; (b) a cleanup task that must run even if the load failed; (c) a join where every upstream must have succeeded and you want to fail loudly otherwise; (d) a join where at least one upstream must have succeeded; (e) an alert task that should fire only if something failed; (f) a task that must run after every upstream has finished, successful or not.

2. **Harder — convert the branch to dynamic mapping.** Take the `patterns_branch` DAG's `full_load` / `incremental_load` branch and replace it with a dynamically-mapped `load_region` task that expands over the regions present in the data. Write the join with the correct trigger rule, and add an assertion that the join runs when one mapped instance fails and when all of them succeed.

3. **Harder — prove the loop anti-pattern is worse than it looks.** Write a DAG file that generates 50 tasks with a loop, and a second that generates one mapped task. Compare the number of task instances created when the data contains 4 regions, and explain what each DAG's scheduler load looks like at parse time.

4. **Harder — design a Dataset-based producer/consumer pair.** Write two DAGs: `orders_producer` (which writes a partition and updates the dataset) and `orders_consumer` (scheduled on the dataset). Then answer three questions in writing: (a) how does the consumer learn the producer's logical date? (b) what happens if the producer fails silently for a week? (c) how would you prevent the consumer from running twice for one producer update?

5. **Harder — make the backfill safe under concurrency.** A backfill of 30 dates runs with `max_active_runs=5`. Two dates write to the same partition decorator. Explain exactly how that happens, what the symptom would be in the output table, and which two settings prevent it. Then explain why raising `max_active_runs` to speed up the backfill makes the problem *more* likely, not less.

6. **Hardest — audit the course's DAGs against the pattern catalogue.** For each DAG in `project/dags/`, name the patterns it demonstrates and identify the one pattern each is *missing* that a production version would need. Then pick the DAG with the biggest gap and sketch the improvement.

<details>
<summary><b>Solution (click to reveal)</b></summary>

**1.**

| Scenario | Trigger rule | Why |
|---|---|---|
| (a) | `NONE_FAILED_MIN_ONE_SUCCESS` | With `all_success` the join never runs when the *un-taken* branch is skipped — the classic "my downstream task silently never runs" bug. With `none_failed` the join runs even if every branch was skipped, producing an empty result. |
| (b) | `all_done` | "Run regardless of what happened upstream" — a skipped or failed dependency does not stop it. |
| (c) | `all_success` (the default) | The whole point is to fail loudly when an upstream did not succeed. |
| (d) | `NONE_FAILED_MIN_ONE_SUCCESS` | Requires at least one success, so an all-skipped join is skipped rather than running with no data. |
| (e) | `one_failed` | Fires on the failure, not on success. |
| (f) | `all_done` | Same as (b): run once every upstream has finished, successful or not. |

The six scenarios reduce to four distinct rules. You only ever need four.

**2.**

The fan-out comes from `regions_to_process(summary)`; the mapped task is `PythonOperator.partial(task_id="load_region", python_callable=load_one_region).expand(region=regions.output)`; the join uses `TriggerRule.NONE_FAILED_MIN_ONE_SUCCESS`.

The key point about the assertions: the join's trigger rule decides whether it runs *at all*. With `NONE_FAILED_MIN_ONE_SUCCESS`, a join whose upstreams are one success and one failure still runs — and receives only the successful instance's return value, because the mapped task's collected list contains only the instances that ran. So the two assertions are:

```python
# one instance fails -> the join still runs, with the partial result
assert run.state["join_regions"] == "success"
assert run.pull("join_regions")["joined"] == 3      # 4 regions, 1 failed

# all instances succeed -> the join runs with the full result
assert run.state["join_regions"] == "success"
assert run.pull("join_regions")["joined"] == 4
```

With the default `all_success` the first assertion fails with `upstream_failed` — which is precisely the bug the trigger rule exists to prevent.

**3.**

The loop version creates **50 task instances in every run**, regardless of data. All 50 appear in the UI, all 50 occupy rows in the metadata database, and all 50 are re-parsed by the scheduler on every parse cycle — for every DAG in the environment, since parsing is per-file but the scheduler holds the resulting graph in memory.

The mapped version creates **4** — the number the data actually implies. Its parse-time cost is constant: one task definition.

So the loop version's scheduler load is proportional to the number of items *in the code*, and it grows every time someone adds an item. The mapped version's is proportional to the number of items *in the data*, and it costs the same parse time either way.

**4.**

(a) The consumer learns the producer's date through dataset **event metadata** (Airflow 2.9+): `outlets=[Metadata(ORDERS_DATASET, extra={"logical_date": "{{ ds }}"})]`. The consumer reads it from the event. The fallback, on older Airflow, is to pass it in `params`. The critical fact is that **the consumer's own logical date is the time it was queued**, not the producer's — a dataset-triggered run is not tied to the producer's date.

(b) The consumer simply **never runs**, and nothing reports it. This is the gap Datasets cannot close: a Dataset has no timeout, because a dataset has no notion of "expected". The answer is a separate monitoring DAG (or a freshness check on the producer's artefact) that alerts when the producer's last successful run is older than it should be — *not* a sensor, because a sensor would reintroduce the polling and the coupling the Dataset was chosen to avoid.

(c) Make the consumer **idempotent** and give it a **deterministic destination** keyed on the producer's logical date. A duplicate trigger then writes the same partition again, and the idempotent write (truncate, or dedup on the business key) makes the second run a no-op. `dedup_key(order_id, partition)` exists for exactly this: the identity is derived from the data, so a re-delivered update resolves to the same row.

**5.**

The symptom is **doubled revenue for one date** — or a partition whose row count is twice the source. The mechanism: the partition decorator is derived from the run's logical date, so two runs with *different* logical dates write to *different* partitions, and the doubling appears when the partition filter used by the aggregate is wider than the partitions actually loaded. The more precise failure is that the aggregate's `WHERE order_ts >= start AND order_ts < end` window can span a partition that a second, concurrent run is still writing to — so the aggregate reads a partially-written partition and the two runs' contributions overlap.

Two settings prevent it: **`max_active_runs=1`** (one date at a time), and a **deterministic, date-derived destination** (`partition = logical_date`) with an **idempotent write** so a retry replaces rather than appends.

Raising `max_active_runs` makes it *more* likely, not less: more concurrent runs means more windows in which two runs overlap on the same aggregate window, and more opportunity for a partial-partition read. The usual intuition — "more parallelism, same result" — holds only when every run touches a disjoint output region *and* the reader never observes a partially-written region. Those two properties are what `max_active_runs=1` and a closed-window read give you.

**6.**

- **`01_hello_world.py`** — linear only; a conditional check plus an SLA. It would also need a *load* pattern and a `max_active_runs` policy.
- **`02_templating_and_variables.py`** — linear; it would need a downstream handoff (a `TriggerDagRunOperator` or a Dataset consumer) and a `params` contract.
- **`03_etl_bigquery.py`** — a linear pipeline with an SLA, a retry policy, an idempotent partitioned load, a dead-letter path and `depends_on_past=False`. It is missing **fan-out** (per-region loads) and a **cross-DAG handoff**. Its aggregate's `WHERE order_ts >= ... AND order_ts < ...` filter is exactly right, which is the hardest thing to get right.
- **`04_branching_and_xcom.py`** — conditional execution, XCom, a sensor, a cross-DAG trigger, and a join with `none_failed`. It is missing **dynamic mapping** (the branch's two fixed tasks could be a mapped task over the load strategies) and a **Dataset-based alternative to the sensor**.

**The biggest gap is `03_etl_bigquery.py`.** It is the production-shaped DAG, and it is the one most likely to be operated on daily — yet it has no fan-out and no cross-DAG handoff, so a late-arriving partition has nowhere to wait and a downstream consumer has to poll for it.

The improvement, in order:

1. Replace the single `load_to_bigquery` with a mapped task over the partitions present in the run, using `expand_kwargs(load_tasks(...))`. The load becomes per-partition, and a failed partition retries on its own instead of re-running the whole load.
2. Add a `GcsObjectSensor` (or a Dataset) at the head of the DAG, so the run waits for the source object rather than proceeding with an empty load. The course's `source_is_ready` is the pure form of the same condition.
3. Give the aggregate a `NONE_FAILED_MIN_ONE_SUCCESS` join after the mapped load, so a partial load still publishes — with the failure visible in the log — instead of silently producing nothing.
4. Split the export into its own DAG triggered by a `TriggerDagRunOperator`, so the downstream consumer of the exported summary is not coupled to the load's internals.

</details>